# Q1 — Pretrained models (ResNet18, DenseNet121, VGG19)

Train 15-class classifiers using pretrained backbones. Compute per-class precision and recall.
Adjust hyperparameters and run on GPU if available.

In [3]:
# Imports and dataset discovery
import os, re, math, random
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import precision_recall_fscore_support
import matplotlib.pyplot as plt

# Set dataset root (adjust if your dataset folder is named differently)
DATA_ROOT = Path('/Users/sunnel/Desktop/LLMs and GenAI Assignment/Datasets/dataset')
print('DATA_ROOT:', DATA_ROOT)
# If dataset is structured as class subfolders, list classes
if DATA_ROOT.exists() and any(p.is_dir() for p in DATA_ROOT.iterdir()):
    CLASSES = sorted([p.name for p in DATA_ROOT.iterdir() if p.is_dir()])
else:
    raise RuntimeError('Cannot find dataset folders under Datasets/dataset or Datasets/dataset2/images')
NUM_CLASSES = len(CLASSES)
print('Found classes:', NUM_CLASSES)

# Utility: build train/test split per instructions (0001-0040 -> train; remaining -> test)
def build_splits(root, classes):
    train_list = []
    test_list = []
    num_re = re.compile(r'(\d+)')
    for idx, cls in enumerate(classes):
        cls_dir = Path(root)/cls
        if not cls_dir.exists():
            continue
        imgs = sorted([p for p in cls_dir.iterdir() if p.suffix.lower() in ['.jpg','.jpeg','.png']])
        for p in imgs:
            m = num_re.search(p.stem)
            if not m:
                # fallback: use filename order
                if len(train_list) < 40:
                    train_list.append((str(p), idx))
                else:
                    test_list.append((str(p), idx))
            else:
                n = int(m.group(1))
                if 1 <= n <= 40:
                    train_list.append((str(p), idx))
                else:
                    test_list.append((str(p), idx))
    return train_list, test_list

train_items, test_items = build_splits(DATA_ROOT, CLASSES)
print('Train images:', len(train_items), 'Test images:', len(test_items))

# Simple Dataset wrapper
class SimpleImageDataset(Dataset):
    def __init__(self, items, transform=None):
        self.items = items
        self.transform = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        p, label = self.items[idx]
        img = Image.open(p).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

# Transforms and dataloaders
IMG_SIZE = 224
train_tf = transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
test_tf  = transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
train_ds = SimpleImageDataset(train_items, transform=train_tf)
test_ds  = SimpleImageDataset(test_items, transform=test_tf)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=4)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=4)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device', device)

DATA_ROOT: /Users/sunnel/Desktop/LLMs and GenAI Assignment/Datasets/dataset


RuntimeError: Cannot find dataset folders under Datasets/dataset or Datasets/dataset2/images

In [ ]:
# Training & evaluation helper (keeps things simple and configurable)
import time
from tqdm import tqdm

def make_model(name, num_classes, pretrained=True):
    if name == 'resnet18':
        m = models.resnet18(pretrained=pretrained)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == 'densenet121':
        m = models.densenet121(pretrained=pretrained)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    elif name == 'vgg19':
        m = models.vgg19(pretrained=pretrained)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
    else:
        raise ValueError('Unknown model')
    return m.to(device)

def train_model(model, train_loader, epochs=5, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    model.train()
    for e in range(epochs):
        loop = tqdm(train_loader, desc=f'Epoch {e+1}/{epochs}')
        for xb, yb in loop:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            out = model(xb)
            loss = crit(out, yb)
            loss.backward()
            opt.step()
            loop.set_postfix(loss=loss.item())

def evaluate(model, loader):
    model.eval()
    y_true = []
    y_pred = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            y_pred.extend(preds.tolist())
            y_true.extend(yb.numpy().tolist())
    return np.array(y_true), np.array(y_pred)

def run_and_report(model_name, epochs=3):
    print('Running', model_name)
    m = make_model(model_name, NUM_CLASSES, pretrained=True)
    train_model(m, train_loader, epochs=epochs)
    y_true, y_pred = evaluate(m, test_loader)
    prec, rec, f1, sup = precision_recall_fscore_support(y_true, y_pred, labels=list(range(NUM_CLASSES)), zero_division=0)
    import pandas as pd
    df = pd.DataFrame({'class': CLASSES, 'precision': prec, 'recall': rec, 'f1': f1, 'support': sup})
    display(df)
    return df

# Example runs (comment/uncomment as needed)
# df_resnet = run_and_report('resnet18', epochs=2)
# df_densenet = run_and_report('densenet121', epochs=2)
# df_vgg = run_and_report('vgg19', epochs=2)

print('Notebook ready — run the example runs above, adjust epochs for full training.')